In [1]:
import pandas as pd
from mod06_deanonymize import load_data, link_records, deanonymization_rate

In [2]:
anon_path = "mod06_data/anonymized.csv"
aux_path = "mod06_data/auxiliary.csv"

### Load and review the data

In [3]:
anon, aux = load_data(anon_path, aux_path)

In [4]:
anon.head(20)

,anon_id,age,zip3,gender
0,ERM5C,50,432,F
1,OFJJP,75,323,F
2,I3H24,44,323,M
3,3WUYW,66,323,M
4,ZN91P,18,323,M
5,7BLKV,36,606,F
6,4SJFH,70,890,M
7,0NO3L,31,110,M
8,U060Z,80,191,F
9,1761H,58,606,M


### Merge the datasets by age, gender, and zip3 and return the unique matches

#### Explore candidate matches
Merge the two tables using exact matches on age, gender, and zip3. An inner merge keeps only matching combinations. If several auxiliary records match one anonymous ID, that ID appears in several rows. These are candidate matches; next we will keep only IDs with exactly one candidate.

In [5]:
candidates = anon.merge(
    aux,
    on=["age", "gender", "zip3"],
    how="inner"
)

candidates.head(10)

,anon_id,age,zip3,gender,name
0,ERM5C,50,432,F,Grace Lopez
1,OFJJP,75,323,F,Grace Martinez
2,OFJJP,75,323,F,Emma Jones
3,I3H24,44,323,M,Thomas Moore
4,I3H24,44,323,M,Paul Taylor
5,3WUYW,66,323,M,Paul Martin
6,3WUYW,66,323,M,David Taylor
7,ZN91P,18,323,M,Kyle Martinez
8,7BLKV,36,606,F,Emma Garcia
9,7BLKV,36,606,F,Hannah Smith


#### Keep only unique matches
Group the candidate rows by anon_id. transform("size") assigns each row the number of candidates for its anonymous ID. Keep rows with a count of exactly one; IDs with multiple candidates are ambiguous. Select anon_id and name, then rename name to matched_name to match the function's required output.

In [6]:
candidate_counts = candidates.groupby("anon_id")["name"].transform("size")

unique_matches = candidates.loc[
    candidate_counts == 1,
    ["anon_id", "name"]
].rename(columns={"name": "matched_name"})

unique_matches.head(10)

,anon_id,matched_name
0,ERM5C,Grace Lopez
7,ZN91P,Kyle Martinez
14,0NO3L,Marcus Lopez
15,U060Z,Sally Davis
16,1761H,Frank Jackson
23,ULMOZ,David Davis
30,Q1J6V,Marcus Jones
31,BU6DU,Sally Rodriguez
32,LL1K7,Omar Jackson
37,SJIYY,Grace Gonzalez


#### Use the completed matching function
The merge and filtering steps now live in link_records() in mod06_deanonymize.py. Calling link_records(anon, aux) passes the two tables into its anon_df and aux_df parameters. The function returns unique_matches, which the cell below stores in matches.

After updating the Python file, restart the notebook kernel and run the cells from the top so Python imports the updated functions.

In [7]:
matches = link_records(anon, aux)
matches.head()

,anon_id,matched_name
0,ERM5C,Grace Lopez
7,ZN91P,Kyle Martinez
14,0NO3L,Marcus Lopez
15,U060Z,Sally Davis
16,1761H,Frank Jackson


### Calculate the match rate
The match rate is the number of uniquely matched records divided by the total number of original anonymous records. In deanonymization_rate(), len(matches_df) counts unique matches and len(anon_df) counts all anonymous records. The function returns 0.0 for an empty anonymous table to avoid division by zero.

The first output is the fraction between 0 and 1. The second formats that fraction as a percentage with one decimal place.

In [8]:
rate = deanonymization_rate(matches, anon)
print(rate)
print(f"Uniquely matched: {rate:.1%}")

0.421
Uniquely matched: 42.1%


# Discussion Questions

### Describe your reaction to the match rate. What sources of data might be prone to privacy issues?

I assumed that the data could be matched to specific people, but I was surprised by how well it worked. About 42% of the anonymous records were linked to a single named candidate using only age, gender, and the first three digits of a ZIP code. This shows that removing names alone may not adequately protect privacy. Details that seem ordinary can help identify someone when combined with another dataset.

Medical records, surveys, and location data can put people’s privacy at risk if someone compares them with other outside information to figure out who the records belong to.

### Provide examples of at least two other variables that could increase the match rate if included.

Two additional variables that could increase the match rate are full date of birth and street address. A full date of birth could help distinguish people who are the same age but have different birthdays. A street address could help because people with the same age, gender, and ZIP prefix may live at different addresses. If these details were available in both datasets, they could reduce ambiguous matches and help link more anonymous records to a single named candidate.